Please use this notebook to perform feature engineering and feature dataset analysis. In this section:
- Clean data based on bronze report
- Build recommended features (as well as any others) from analysis report
- Re-run dataset analysis

In [0]:
!pip install --force-reinstall --no-deps /Volumes/workspace/ds_cb_wheels/wheels/eda_toolkit-0.1.0-py3-none-any.whl

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import os 
import scipy
from pyspark.sql import SparkSession
from eda_toolkit.first_stage_report_building.ops.generate_report import build_report_regression

In [0]:
spark = SparkSession.builder.getOrCreate()
df_bronze = spark.sql("""
                      select * from workspace.ds_cb_wind_turbine.bronze
                      """).toPandas()

In [0]:
print(df_bronze.loc[df_bronze['lv_active_power'] < 0]['lv_active_power'].describe())

In [0]:
df_bronze.drop(columns = ['theoretical_power_curve'], inplace = True)

In [0]:
df_bronze['lv_active_power'] = df_bronze['lv_active_power'].clip(lower = 0)

In [0]:
df_bronze['hour'] = pd.to_datetime(df_bronze['timestamp']).dt.hour
df_bronze['monr'] = pd.to_datetime(df_bronze['timestamp']).dt.hour

In [0]:
print(df_bronze['timestamp'].min())
print(df_bronze['timestamp'].max())

In [0]:
df_bronze.head()

In [0]:
df_bronze['sine_hour'] = np.sin(2 * np.pi * df_bronze['hour']/24)
df_bronze['cosine_hour'] = np.cos(2 * np.pi * df_bronze['hour']/24)

df_bronze = df_bronze.sort_values(by = ['timestamp'])

In [0]:
df_bronze['wind_direction_rad'] = np.deg2rad(df_bronze['wind_direction'])
df_bronze['wind_magnitude_i'] = df_bronze['wind_speed'] * (np.cos(df_bronze['wind_direction_rad']))
df_bronze['wind_magnitude_j'] = df_bronze['wind_speed'] * (np.sin(df_bronze['wind_direction_rad']))
df_bronze['wind_speed_x3'] = df_bronze['wind_speed']**3

In [0]:
df_bronze['wind_magnitude_i_3h_rolling_mean'] = df_bronze['wind_magnitude_i'].rolling(window = 3).mean()
df_bronze['wind_magnitude_j_3h_rolling_mean'] = df_bronze['wind_magnitude_j'].rolling(window = 3).mean()
df_bronze['wind_magntitue_i_3h_rolling_std'] = df_bronze['wind_magnitude_i'].rolling(window = 3).std()
df_bronze['wind_magntitue_j_3h_rolling_std'] = df_bronze['wind_magnitude_j'].rolling(window = 3).std()
df_bronze['wind_speed_3h_rolling_mean'] = df_bronze['wind_speed'].rolling(window = 3).mean()
df_bronze['wind_speed_3h_rolling_std'] = df_bronze['wind_speed'].rolling(window = 3).std()
df_bronze['wind_direction_3h_rolling_mean'] = df_bronze['wind_direction'].rolling(window = 3).mean()
df_bronze['wind_direction_3h_rolling_std'] = df_bronze['wind_direction'].rolling(window = 3).std()
df_bronze['wind_speed_x3_3h_rolling_mean'] = df_bronze['wind_speed_x3'].rolling(window = 3).mean()
df_bronze['wind_speed_x3_3h_rolling_std'] = df_bronze['wind_speed_x3'].rolling(window = 3).std()

df_bronze.dropna(inplace=True)

build_report_regression(df_bronze, 'lv_active_power', ['timestamp'])

df_silver = spark.createDataFrame(df_bronze)

spark.sql("""
          drop table if exists workspace.ds_cb_wind_turbine.silver
          """)
df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.ds_cb_wind_turbine.silver")